# 03 — Parse and store

**Goal.** Turn the raw HTML from notebook 02 into clean text, drop near-duplicates, and persist a queryable corpus in DuckDB. This is the substrate every later notebook reads from.

**Inputs.** `data/raw_html/*.html` + `data/raw_html/_index.jsonl` (from nb 02).

**Output.** `db/cti.duckdb` with a `documents` table:

| column | type | note |
|--------|------|------|
| url_id | TEXT PK | sha1(url) |
| url | TEXT | original onion URL |
| source | TEXT | 'onion' or 'telegram' (nb 05 also writes here) |
| title | TEXT | extracted page title |
| text | TEXT | trafilatura-cleaned body |
| n_chars | INT | len(text) |
| fetched_at | TIMESTAMP | from nb 02 index |
| dedup_group | INT | shared by near-duplicates; smallest url_id wins |
| label | TEXT | filled by nb 04 |
| score | DOUBLE | filled by nb 04 |

In [1]:
import json
from pathlib import Path
import duckdb
import trafilatura
from datasketch import MinHash, MinHashLSH

RAW_DIR = Path("data/raw_html")
DB_PATH = Path("db/cti.duckdb")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

## 1. Schema

Notebook 05 (Telegram) writes into the same table with `source='telegram'`. Notebook 04 updates `label` and `score`. Notebook 06 reads everything.

In [2]:
con = duckdb.connect(str(DB_PATH))
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    url_id      TEXT PRIMARY KEY,
    url         TEXT,
    source      TEXT,
    title       TEXT,
    text        TEXT,
    n_chars     INTEGER,
    fetched_at  TIMESTAMP,
    dedup_group INTEGER,
    label       TEXT,
    score       DOUBLE
);
""")
print("table ready")

table ready


## 2. Load the fetch index

In [3]:
index_path = RAW_DIR / "_index.jsonl"
records = [json.loads(l) for l in index_path.read_text().splitlines() if l.strip()]
successes = [r for r in records if r.get("status") == 200]
print(f"{len(successes)} successful fetches out of {len(records)} attempts")

4 successful fetches out of 5 attempts


## 3. Extract clean text with trafilatura

trafilatura strips boilerplate (nav, footer, ads) and returns the main body. We also pull the page title from its metadata.

In [4]:
def extract(html: str):
    text = trafilatura.extract(html, include_comments=False, include_tables=False) or ""
    meta = trafilatura.extract_metadata(html)
    title = (meta.title if meta and meta.title else "").strip()
    return title, text.strip()

parsed = []
for r in successes:
    html_path = RAW_DIR / f"{r['url_id']}.html"
    if not html_path.exists():
        continue
    title, text = extract(html_path.read_text(encoding="utf-8", errors="ignore"))
    if len(text) < 200:  # too short to be useful
        continue
    parsed.append({
        "url_id": r["url_id"],
        "url": r["url"],
        "source": "onion",
        "title": title,
        "text": text,
        "n_chars": len(text),
        "fetched_at": r["fetched_at"],
    })

print(f"{len(parsed)} parsed documents (>=200 chars)")

4 parsed documents (>=200 chars)


## 4. Near-duplicate detection with MinHash

Many onion sites are mirrors of each other. We compute a 128-permutation MinHash over 5-word shingles and group documents with Jaccard similarity above 0.8.

In [5]:
def shingles(text: str, k: int = 5):
    words = text.lower().split()
    return {" ".join(words[i:i+k]) for i in range(max(0, len(words) - k + 1))}

def minhash(text: str, num_perm: int = 128):
    m = MinHash(num_perm=num_perm)
    for s in shingles(text):
        m.update(s.encode())
    return m

lsh = MinHashLSH(threshold=0.8, num_perm=128)
hashes = {}
for d in parsed:
    h = minhash(d["text"])
    hashes[d["url_id"]] = h
    lsh.insert(d["url_id"], h)

# Greedy union-find: assign each doc a group id (smallest url_id in its dup-cluster)
group = {}
for d in parsed:
    uid = d["url_id"]
    if uid in group:
        continue
    cluster = lsh.query(hashes[uid])  # includes self
    leader = min(cluster)
    for c in cluster:
        group[c] = leader

# Assign integer dedup_group ids
leader_to_int = {leader: i for i, leader in enumerate(sorted(set(group.values())))}
for d in parsed:
    d["dedup_group"] = leader_to_int[group[d["url_id"]]]

n_groups = len(leader_to_int)
print(f"{len(parsed)} docs collapse to {n_groups} dedup groups")

4 docs collapse to 4 dedup groups


## 5. Persist to DuckDB

We use `INSERT OR REPLACE` so re-running this notebook overwrites stale rows without dropping the table.

In [6]:
rows = [(
    d["url_id"], d["url"], d["source"], d["title"], d["text"],
    d["n_chars"], d["fetched_at"], d["dedup_group"], None, None,
) for d in parsed]

con.executemany("""
    INSERT OR REPLACE INTO documents
    (url_id, url, source, title, text, n_chars, fetched_at, dedup_group, label, score)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", rows)

n = con.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
print(f"documents table now has {n} rows")

documents table now has 4 rows


## 6. Quick sanity check

In [7]:
con.execute("""
    SELECT source, COUNT(*) AS n_docs, COUNT(DISTINCT dedup_group) AS n_groups,
           ROUND(AVG(n_chars)) AS avg_chars
    FROM documents
    GROUP BY source
""").fetchdf()

,source,n_docs,n_groups,avg_chars
0,onion,4,4,1190.0


## 7. Inspect a single stored row

Pull one document end-to-end so you can eyeball what trafilatura actually saved.

In [8]:
# Pick the longest doc so the preview is meaningful
row = con.execute("""
    SELECT url_id, url, source, title, n_chars, fetched_at, dedup_group, text
    FROM documents
    ORDER BY n_chars DESC
    LIMIT 1
""").fetchone()

(url_id, url, source, title, n_chars, fetched_at, dedup_group, text) = row

print(f"url_id      : {url_id}")
print(f"url         : {url}")
print(f"source      : {source}")
print(f"title       : {title!r}")
print(f"n_chars     : {n_chars}")
print(f"fetched_at  : {fetched_at}")
print(f"dedup_group : {dedup_group}")
print(f"\n--- text (first 1000 chars) ---\n{text[:1000]}")
print(f"\n--- text (last 300 chars) ---\n{text[-300:]}")

url_id      : 770f1d9e993c747bc59f1484e6eea49d7832d37a
url         : http://bbcnewsd73hkzno2ini43t4gblxvycyac5aw4gnv7t2rccijh7745uqd.onion/
source      : onion
title       : 'BBC - Home'
n_chars     : 2253
fetched_at  : 2026-04-29 15:10:54.935359
dedup_group : 2

--- text (first 1000 chars) ---
BBC Homepage
Live.Â
Police declare terrorist incident after two Jewish men stabbed in north London
The 45-year-old suspect has a history of serious violence and mental health issues, police say. They also say he attempted to stab officers while being arrested.
- Attribution
King's US visit
Keep up with the latest from BBC Sport
The top stories from England, Scotland, Wales and Northern Ireland
- Attribution
- Comments
- AttributionNews
Latest news and must-see moments
Richard Gadd paints a picture of masculinity in BBC drama Half Man
'It feels like the debate about men has reached quite a high pitch.'
- AttributionBBC Writers
- AttributionBBC Bitesize
Meal ideas, cooking tips and more, updated d

## What's next

Notebook 04 loads the DarkBERT classifier you trained in cti_301/nb 04, runs fp16 inference over the deduplicated documents (one per `dedup_group`), and writes `label` + `score` back into this same table.